# CodeLlama Automatic Evaluation

In [ ]:

from pathlib import Path
import json
import os
import random

import nltk
import numpy as np
import pandas as pd
import torch
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
MAX_INPUT_TOKENS = 12000
MAX_NEW_TOKENS = 128
DATASET_FILE = Path("evaluation_dataset.jsonl")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

smooth = SmoothingFunction().method1
rouge_scorer_instance = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)


def load_dataset():
    records = []
    with DATASET_FILE.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def format_context(record):
    commit_text = "\n".join(
        f"{item['hash']} | {item['date']} | {item['subject']}"
        for item in record["commits"]
    )

    if not commit_text:
        commit_text = "(none)"

    return f"""Repository: {record['repository']}
Repository Commit: {record['repository_sha']}
File: {record['file_name']}

Code:
{record['source_code']}

Code Size:
Lines: {record['code_size']['lines']}
Nonblank Lines: {record['code_size']['nonblank_lines']}
Characters: {record['code_size']['characters']}
Bytes: {record['code_size']['bytes']}
NLOC: {record['code_size']['nloc']}

Comments:
{record['comments'] or '(none)'}

README ({record['readme_name'] or 'none'}):
{record['readme_content'] or '(none)'}

Complexity:
Function Count: {record['complexity']['function_count']}
Mean Cyclomatic Complexity: {record['complexity']['mean_ccn']}
Maximum Cyclomatic Complexity: {record['complexity']['max_ccn']}

File Commit History:
{commit_text}
"""


def prompt_zero(record):
    return f"""Generate a concise code summary from the following file and repository context. Describe the main purpose, behavior, important inputs or outputs, and relevant dependencies. Do not reproduce the source code or raw metadata.

{format_context(record)}

Output only the final summary.
"""


def prompt_few(record):
    return f"""Generate a concise code summary from the following file and repository context.

Example input:
Code:
def load_config(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path) as file:
        return json.load(file)

README:
Configuration utility.

Complexity:
Function Count: 1
Mean Cyclomatic Complexity: 2
Maximum Cyclomatic Complexity: 2

File Commit History:
abc123 | 2025-01-01 | Add missing-file validation

Example output:
The `load_config` function reads a JSON configuration file and returns the parsed settings. It validates that the requested file exists and raises `FileNotFoundError` when the file cannot be found.

Actual input:
{format_context(record)}

Output only the final summary.
"""


def prompt_adv(record):
    return f"""Analyze the following file and repository context step by step before producing the final code summary. Identify the file's purpose, main operations, inputs and outputs, dependencies, and relevant contextual information. Do not show the reasoning process.

{format_context(record)}

Output only the final summary.
"""


def metric_scores(prediction, reference):
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()

    bleu = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        smoothing_function=smooth,
    )
    rouge_l = rouge_scorer_instance.score(
        reference,
        prediction,
    )["rougeL"].fmeasure
    meteor = meteor_score(
        [reference_tokens],
        prediction_tokens,
    )

    return round(bleu, 6), round(rouge_l, 6), round(meteor, 6)


def prompt_token_count(prompt):
    messages = [
        {
            "role": "system",
            "content": "You are an expert software engineer specializing in code summarization.",
        },
        {"role": "user", "content": prompt},
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return len(
        tokenizer(
            formatted,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )


def generate_summary(prompt):
    messages = [
        {
            "role": "system",
            "content": "You are an expert software engineer specializing in code summarization.",
        },
        {"role": "user", "content": prompt},
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=False,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][input_length:]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


In [ ]:

hf_token = os.getenv("HF_TOKEN")
model_name = "codellama/CodeLlama-7b-Instruct-hf"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

model.eval()


In [ ]:

records = load_dataset()
results = []

for record in tqdm(records, desc="CodeLlama"):
    prompts = {
        "Zero": prompt_zero(record),
        "Few": prompt_few(record),
        "Adv": prompt_adv(record),
    }

    token_counts = {
        name: prompt_token_count(prompt)
        for name, prompt in prompts.items()
    }

    max_tokens = max(token_counts.values())

    if max_tokens > MAX_INPUT_TOKENS:
        results.append({
            "File ID": record["file_id"],
            "Repository": record["repository"],
            "Repository SHA": record["repository_sha"],
            "File Name": record["file_name"],
            "File URL": record["file_url"],
            "Skipped": True,
            "Skip Reason": f"Context requires {max_tokens} tokens.",
            "Context Tokens Zero": token_counts["Zero"],
            "Context Tokens Few": token_counts["Few"],
            "Context Tokens Adv": token_counts["Adv"],
        })
        continue

    reference = record["reference_summary"]

    zero_summary = generate_summary(prompts["Zero"])
    few_summary = generate_summary(prompts["Few"])
    adv_summary = generate_summary(prompts["Adv"])

    bz, rz, mz = metric_scores(zero_summary, reference)
    bf, rf, mf = metric_scores(few_summary, reference)
    ba, ra, ma = metric_scores(adv_summary, reference)

    results.append({
        "File ID": record["file_id"],
        "Repository": record["repository"],
        "Repository SHA": record["repository_sha"],
        "File Name": record["file_name"],
        "File URL": record["file_url"],
        "Code Lines": record["code_size"]["lines"],
        "Code Characters": record["code_size"]["characters"],
        "NLOC": record["code_size"]["nloc"],
        "Function Count": record["complexity"]["function_count"],
        "Mean CCN": record["complexity"]["mean_ccn"],
        "Max CCN": record["complexity"]["max_ccn"],
        "Comment Characters": len(record["comments"]),
        "README Characters": len(record["readme_content"]),
        "Commit Count": len(record["commits"]),
        "Reference Summary": reference,
        "Context Tokens Zero": token_counts["Zero"],
        "Context Tokens Few": token_counts["Few"],
        "Context Tokens Adv": token_counts["Adv"],
        "Skipped": False,
        "Skip Reason": "",
        "Zero Summary": zero_summary,
        "BLEU Zero": bz,
        "ROUGE Zero": rz,
        "METEOR Zero": mz,
        "Few Summary": few_summary,
        "BLEU Few": bf,
        "ROUGE Few": rf,
        "METEOR Few": mf,
        "Adv Summary": adv_summary,
        "BLEU Adv": ba,
        "ROUGE Adv": ra,
        "METEOR Adv": ma,
    })

results_df = pd.DataFrame(results)
results_df.to_excel("CodeLlama_Automatic_Evaluation_Results.xlsx", index=False)
display(results_df.head())
